## Cálculo dos indicadores

Nesse notebook são calculadas as variáveis de percentual de votos dos candidatos e de famílias beneficiárias do Programa Bolsa Família (PBF)

In [1]:
from utils.load_csv import load_csv
from utils.save_csv import save_csv

In [2]:
df_resultados_2026_filtrado = load_csv('resultados_2026_filtrado.csv')

secoes_cad = load_csv("secoes_cadastro.csv")

locais_votacao = load_csv("locais_votacao.csv")

df_observasampa_filtrado = load_csv('observasampa_filtrado.csv')

1 - Votos válidos dos candidatos à presidência no segundo turno por local de votação do município de São Paulo

In [3]:
VOTO_BRANCO, VOTO_NULO = 95, 96
CANDIDATOS = {
    "FLAVIO NANTES BOLSONARO": "flavio",
    "LUIZ INÁCIO LULA DA SILVA": "lula",
}

df = df_resultados_2026_filtrado.copy()

assert df["NR_TURNO"].eq(2).all()
assert df["DS_CARGO"].nunique() == 1
assert df[["NR_ZONA", "NR_SECAO"]].notna().all().all()

df["NR_ZONA"] = df["NR_ZONA"].astype(int)
df["NR_SECAO"] = df["NR_SECAO"].astype(int)

df = df.merge(
    secoes_cad[["zona", "secao", "id_local"]],
    left_on=["NR_ZONA", "NR_SECAO"], right_on=["zona", "secao"],
    how="left", validate="many_to_one",
)
assert df["id_local"].notna().all(), "Seção com votos sem local no cadastro"

eh_especial = df["NR_VOTAVEL"].isin([VOTO_BRANCO, VOTO_NULO])
candidatos = df[~eh_especial]
assert set(candidatos["NM_VOTAVEL"].unique()) == set(CANDIDATOS), candidatos["NM_VOTAVEL"].unique()

votos = (
    candidatos
    .pivot_table(index="id_local", columns="NM_VOTAVEL", values="QT_VOTOS",
                 aggfunc="sum", fill_value=0)
    .rename(columns={n: f"vt_val_{c}" for n, c in CANDIDATOS.items()})
    .rename_axis(columns=None)
    .reset_index()
)

especiais = (
    df[eh_especial]
    .assign(tipo=lambda d: d["NR_VOTAVEL"].map({VOTO_BRANCO: "vt_branco", VOTO_NULO: "vt_nulo"}))
    .pivot_table(index="id_local", columns="tipo", values="QT_VOTOS",
                 aggfunc="sum", fill_value=0)
    .reindex(columns=["vt_branco", "vt_nulo"], fill_value=0)
    .rename_axis(columns=None)
    .reset_index()
)

cols_cand = [f"vt_val_{c}" for c in CANDIDATOS.values()]
cols_votos = cols_cand + ["vt_branco", "vt_nulo"]

df_votos_local = (
    locais_votacao
    .merge(votos, on="id_local", how="left")
    .merge(especiais, on="id_local", how="left")
)
df_votos_local[cols_votos] = df_votos_local[cols_votos].fillna(0).astype(int)

df_votos_local["vt_validos"] = df_votos_local[cols_cand].sum(axis=1)
df_votos_local["vt_total"] = (df_votos_local["vt_validos"]
                              + df_votos_local["vt_branco"] + df_votos_local["vt_nulo"])

denom = df_votos_local["vt_validos"].where(df_votos_local["vt_validos"] > 0)
for c in CANDIDATOS.values():
    df_votos_local[f"pct_val_{c}"] = df_votos_local[f"vt_val_{c}"] / denom * 100

df_votos_local["local_especial"] = df_votos_local["qt_eleitores"].eq(0)

assert df_votos_local["vt_total"].sum() == df["QT_VOTOS"].sum(), "Votos perdidos na agregação"
assert df_votos_local["id_local"].is_unique

print("Locais sem nenhum voto:", int(df_votos_local["vt_total"].eq(0).sum()))
print("Locais especiais:", int(df_votos_local["local_especial"].sum()))
print("\nLocais não especiais com votos acima dos eleitores:")
acima = df_votos_local[(df_votos_local["vt_total"] > df_votos_local["qt_eleitores"])
                       & ~df_votos_local["local_especial"]]
print(acima[["id_local", "nome_local", "vt_total", "qt_eleitores"]].to_string(index=False))
print("\nTotais:")
print(df_votos_local[cols_votos + ["vt_total"]].sum().to_string())

Locais sem nenhum voto: 0
Locais especiais: 10

Locais não especiais com votos acima dos eleitores:
id_local                      nome_local  vt_total  qt_eleitores
001-1562                FACULDADE PHORTE      2956          2783
259-1104 LICEU PASTEUR UNIDADE VERGUEIRO      4647          4539

Totais:
vt_val_flavio    3097923
vt_val_lula      3864354
vt_branco         104890
vt_nulo           216904
vt_total         7284071


2 - Indicador de famílias beneficiárias do PBF por mil habitantes, por distrito da capital paulista

In [4]:
df_observasampa_filtrado['pbf_mil_hab'] = (df_observasampa_filtrado['familias_pbf'] / df_observasampa_filtrado['pop_total']) * 1000

In [5]:
save_csv(df_votos_local, 'votos_local.csv')

save_csv(df_observasampa_filtrado, 'pbf_distrito.csv')

Base salva em data\votos_local.csv
Base salva em data\pbf_distrito.csv
